In [2]:
import pandas as pd
import numpy as np
import re
from rapidfuzz import process, fuzz

In [3]:
df = df = pd.read_csv(r"Dados\turnê_dados_sujos.csv", encoding='utf-8-sig')

- vizualizando características da tabela

In [4]:
print(df.head())

   Rank  Peak All Time Peak  Actual gross Adjusted gross (in 2022 dollars)  \
0     1     1             2  $780,000,000                     $780,000,000   
1     2     1          7[2]  $579,800,000                     $579,800,000   
2     3  1[4]          2[5]  $411,000,000                     $560,622,615   
3     4  2[7]         10[7]  $397,300,000                     $454,751,555   
4     5  2[4]           NaN  $345,675,146                     $402,844,849   

         Artist                   Tour title    Year(s)  Shows Average gross  \
0  Taylor Swift              The Eras Tour †  2023–2024     56   $13,928,571   
1       Beyoncé       Renaissance World Tour       2023     56   $10,353,571   
2       Madonna  Sticky & Sweet Tour ‡[4][a]  2008–2009     85    $4,835,294   
3          Pink  Beautiful Trauma World Tour  2018–2019    156    $2,546,795   
4  Taylor Swift      Reputation Stadium Tour       2018     53    $6,522,173   

  Ref.  
0  [1]  
1  [3]  
2  [6]  
3  [7]  
4  [8

- Número de linhas e colunas

In [5]:
print(f"Linhas: {df.shape[0]}; Colunas: {df.shape[1]}"
)

Linhas: 20; Colunas: 11


- Valores nulos ou ausentes na tabela

In [6]:
print(df.isnull().sum())

Rank                                 0
Peak                                11
All Time Peak                       14
Actual gross                         0
Adjusted gross (in 2022 dollars)     0
Artist                               0
Tour title                           0
Year(s)                              0
Shows                                0
Average gross                        0
Ref.                                 0
dtype: int64


- Verificando Duplicatas

In [7]:
print(df.duplicated().any().sum())

0


- Verificando os tipos de dados

In [8]:
print(df.dtypes)

Rank                                 int64
Peak                                object
All Time Peak                       object
Actual gross                        object
Adjusted gross (in 2022 dollars)    object
Artist                              object
Tour title                          object
Year(s)                             object
Shows                                int64
Average gross                       object
Ref.                                object
dtype: object


 1. padronizando cabeçalho das colunas


In [9]:
df.columns = df.columns.str.replace('\xa0', ' ', regex=True)
print(df.columns.tolist())

['Rank', 'Peak', 'All Time Peak', 'Actual gross', 'Adjusted gross (in 2022 dollars)', 'Artist', 'Tour title', 'Year(s)', 'Shows', 'Average gross', 'Ref.']


2. Removendo colunas vazias
    - Como a maioria dos valores da coluna peak e All time peak estão vazias, vamos removê-las.

In [10]:
df = df.drop(["Peak","All Time Peak"], axis = 1)

3. Tratando os tipos de dados (Tipo Número)

    3.1. convertendo os tipos de dados e retirando caracteres de dolar (Actual gross)

In [11]:
df["Actual gross"] = (df["Actual gross"]
                      .astype(str)
                      .str.strip()
                      .str.replace(r'[^0-9.]', '', regex=True)
                      .astype(float)
                      )

print(df["Actual gross"])

0     780000000.0
1     579800000.0
2     411000000.0
3     397300000.0
4     345675146.0
5     305158363.0
6     280000000.0
7     257600000.0
8     256084556.0
9     250400000.0
10    229100000.0
11    227400000.0
12    204000000.0
13    200000000.0
14    194000000.0
15    184000000.0
16    170000000.0
17    169800000.0
18    167700000.0
19    150000000.0
Name: Actual gross, dtype: float64


    3.2. convertendo os tipos de dados e retirando caracteres de dolar (Adjusted gross (in 2022 dollars))

In [12]:
df["Adjusted gross (in 2022 dollars)"] = (df["Adjusted gross (in 2022 dollars)"]
                      .astype(str)
                      .str.strip()
                      .str.replace(r'[^0-9.]', '', regex=True)
                      .astype(float)
                      )

print(df["Adjusted gross (in 2022 dollars)"])

0     780000000.0
1     579800000.0
2     560622615.0
3     454751555.0
4     402844849.0
5     388978496.0
6     381932682.0
7     257600000.0
8     312258401.0
9     309141878.0
10    283202896.0
11    295301479.0
12    251856802.0
13    299676265.0
14    281617035.0
15    227452347.0
16    213568571.0
17    207046755.0
18    204486106.0
19    185423109.0
Name: Adjusted gross (in 2022 dollars), dtype: float64


3.3. convertendo os tipos de dados e retirando caracteres de dolar (Average gross)

In [13]:
df["Average gross"] = (df["Average gross"]
                      .astype(str)
                      .str.strip()
                      .str.replace(r'[^0-9.]', '', regex=True)
                      .astype(float)
                      )

print(df["Average gross"])

0     13928571.0
1     10353571.0
2      4835294.0
3      2546795.0
4      6522173.0
5      3467709.0
6      2137405.0
7      6282927.0
8      5226215.0
9      2945882.0
10     1735606.0
11     1118227.0
12     1350993.0
13      615385.0
14     3233333.0
15     1295775.0
16     1734694.0
17     2070732.0
18     1385950.0
19     1744186.0
Name: Average gross, dtype: float64


4. Tratando tipo de dados (Tipo Texto) \
    4.1. Coluna Artist

In [14]:
#Lista de nomes corretos
nomes_corretos = [
    "Taylor Swift",
    "Beyoncé",
    "Madonna",
    "Pink",
    "Celine Dion",
    "Lady Gaga",
    "Katy Perry",
    "Cher"
]

# Função para corrigir nomes
def corrigir_nome(nome):
    if pd.isna(nome):
        return nome
    match, score, _ = process.extractOne(nome, nomes_corretos, scorer=fuzz.token_sort_ratio)
    return match if score > 80 else nome  # só corrige se a similaridade for alta, acima de 80%

df["Artist"] = df["Artist"].apply(corrigir_nome)
print(df["Artist"])


0     Taylor Swift
1          Beyoncé
2          Madonna
3             Pink
4     Taylor Swift
5          Madonna
6      Celine Dion
7             Pink
8          Beyoncé
9     Taylor Swift
10         Beyoncé
11       Lady Gaga
12      Katy Perry
13            Cher
14         Madonna
15            Pink
16       Lady Gaga
17         Madonna
18           Adele
19    Taylor Swift
Name: Artist, dtype: object


- Coluna Tour title (definindo uma função)

In [15]:
def limpar_titulo(texto):
    if pd.isna(texto):
        return texto
    #Remove colchetes e referências como [4][a], [15][16], [d]
    texto = re.sub(r'\[[^\]]*\]', '', texto)
    
    #Remove caracteres não alfabéticos ou de pontuação comuns
    texto = re.sub(r'[^\w\s\-\&\'\!\.\,]', '', texto)

    #Remove múltiplos espaços e limpa extremidades
    texto = re.sub(r'\s+', ' ', texto).strip()

    return texto 

df['Tour title'] = df['Tour title'].apply(limpar_titulo)

print(df['Tour title'])
    

0                       The Eras Tour
1              Renaissance World Tour
2                 Sticky & Sweet Tour
3         Beautiful Trauma World Tour
4             Reputation Stadium Tour
5                       The MDNA Tour
6           Taking Chances World Tour
7                     Summer Carnival
8            The Formation World Tour
9                 The 1989 World Tour
10    The Mrs. Carter Show World Tour
11              The Monster Ball Tour
12               Prismatic World Tour
13     Living Proof The Farewell Tour
14                   Confessions Tour
15          The Truth About Love Tour
16                 Born This Way Ball
17                   Rebel Heart Tour
18                    Adele Live 2016
19                       The Red Tour
Name: Tour title, dtype: object


- coluna years

In [16]:
def separar_anos(texto):
    if pd.isna(texto):
        return pd.Series([None, None])
    
    
    texto = re.sub(r'[^0-9]', ' ', str(texto))
    
    anos = re.findall(r'\b\d{4}\b', texto)
    
    if len(anos) == 0:
        return pd.Series([None, None])
    elif len(anos) == 1:
        return pd.Series([anos[0], anos[0]])  # repete se só tiver um ano
    else:
        return pd.Series([anos[0], anos[-1]])  # pega o primeiro e o último

# aplica a função e cria duas novas colunas
df[['Inicio turne', 'Fim turne']] = df['Year(s)'].apply(separar_anos)

# visualiza as novas colunas
print(df[['Year(s)', 'Inicio turne', 'Fim turne']].head())

     Year(s) Inicio turne Fim turne
0  2023–2024         2023      2024
1       2023         2023      2023
2  2008–2009         2008      2009
3  2018–2019         2018      2019
4       2018         2018      2018


- Eliminando a coluna Ref.

In [17]:
df = df.drop("Ref.",axis=1)

print(df.head())

   Rank  Actual gross  Adjusted gross (in 2022 dollars)        Artist  \
0     1   780000000.0                       780000000.0  Taylor Swift   
1     2   579800000.0                       579800000.0       Beyoncé   
2     3   411000000.0                       560622615.0       Madonna   
3     4   397300000.0                       454751555.0          Pink   
4     5   345675146.0                       402844849.0  Taylor Swift   

                    Tour title    Year(s)  Shows  Average gross Inicio turne  \
0                The Eras Tour  2023–2024     56     13928571.0         2023   
1       Renaissance World Tour       2023     56     10353571.0         2023   
2          Sticky & Sweet Tour  2008–2009     85      4835294.0         2008   
3  Beautiful Trauma World Tour  2018–2019    156      2546795.0         2018   
4      Reputation Stadium Tour       2018     53      6522173.0         2018   

  Fim turne  
0      2024  
1      2023  
2      2009  
3      2019  
4      201

- Criando arquico excel com dados limpos

In [18]:
df.to_csv(r"Dados\dados_tratados_turne.csv", index = True, sep = ";", decimal = ",")